# 10. SQL Subqueries, Correlated Execution & Exists: Beginner Guide

### 📝 Universal SQL Execution Order (All SQL Clauses Combined):
```text
┌─ Complete All-in-One SQL Logical Execution Pipeline (All 12 Clauses) ────────┐
│ 1. WITH (CTEs)       ➔ 2. FROM & JOIN (ON)   ➔ 3. WHERE (Row Filter)         │
│ ➔ 4. GROUP BY        ➔ 5. HAVING (Agg Filter)➔ 6. WINDOW (OVER / Partition)  │
│ ➔ 7. QUALIFY         ➔ 8. SELECT & CASE      ➔ 9. DISTINCT (Dedup)           │
│ ➔ 10. UNION/INTERSECT➔ 11. ORDER BY (Sort)   ➔ 12. LIMIT / OFFSET (Page)     │
└──────────────────────────────────────────────────────────────────────────────┘
```

---

### 📌 Overview & Architectural Context
Welcome to **10. SQL Subqueries, Correlated Execution & Exists**. Subqueries are nested queries embedded inside an outer SQL statement. This notebook covers independent scalar subqueries, multi-value set evaluation (`IN`), row-by-row correlated subqueries, and high-performance existence semi-joins (`EXISTS` / `NOT EXISTS`).

### 📚 Key Concepts Covered in this Notebook:
- [x] 🔹 Independent Scalar Subqueries: Single-Value Predicate Injection
- [x] 🔹 Set-Based Multi-Value Comparisons: `IN (SELECT ...)`
- [x] 🔹 Correlated Subqueries: Row-by-Row Outer State Evaluation
- [x] 🔹 Existence Semi-Joins: `EXISTS` & `NOT EXISTS`
- [x] 🔍 Scenario: Identifying Top 1% Spenders Above Regional Benchmark Averages











In [1]:
# Setup in-memory SQLite relational engine with Native SQL Studio Execution
import sqlite3
import pandas as pd
import os
from IPython import get_ipython
from IPython.core.magic import register_line_cell_magic

conn = sqlite3.connect(':memory:')

def load_table(name, path):
    if os.path.exists(path):
        df = pd.read_csv(path)
        df.to_sql(name, conn, index=False, if_exists='replace')

load_table('transactions', 'data/raw_transactions.csv' if os.path.exists('data/raw_transactions.csv') else '../data/raw_transactions.csv')
load_table('customers', 'data/customers.csv' if os.path.exists('data/customers.csv') else '../data/customers.csv')
load_table('merchants', 'data/merchants.csv' if os.path.exists('data/merchants.csv') else '../data/merchants.csv')
load_table('disputes', 'data/disputes.csv' if os.path.exists('data/disputes.csv') else '../data/disputes.csv')

def _execute_raw_sql(query):
    query = query.strip()
    if query.upper().startswith(('INSERT', 'UPDATE', 'DELETE', 'CREATE', 'DROP', 'ALTER', 'VACUUM', 'ANALYZE', 'BEGIN', 'COMMIT', 'ROLLBACK', 'SAVEPOINT')):
        cur = conn.cursor()
        cur.executescript(query)
        conn.commit()
        return "Query Executed Successfully."
    else:
        return pd.read_sql_query(query, conn)

# Register automatic raw SQL transformer & %%sql magic
ip = get_ipython()
if ip is not None:
    def raw_sql_transformer(lines):
        clean_text = ''.join(lines).strip()
        first_token = clean_text.split()[0].upper() if clean_text.split() else ''
        sql_keywords = {'SELECT', 'WITH', 'INSERT', 'UPDATE', 'DELETE', 'CREATE', 'DROP', 'ALTER', 'EXPLAIN', 'ANALYZE', 'VACUUM', 'BEGIN', 'COMMIT', 'ROLLBACK'}
        if first_token in sql_keywords:
            return [f'_execute_raw_sql("""{clean_text}""")']
        return lines
    
    if raw_sql_transformer not in ip.input_transformers_cleanup:
        ip.input_transformers_cleanup.append(raw_sql_transformer)

@register_line_cell_magic
def sql(line, cell=None):
    return _execute_raw_sql(cell if cell is not None else line)

print("SQL Studio Environment Active! You can now write and run pure SQL queries directly.")


SQL Studio Environment Active! You can now write and run pure SQL queries directly.


### 🔹 Independent Scalar Subqueries
- **What it does:** Evaluates an inner query that returns exactly one row and one column, injecting that scalar value into the outer query's comparison predicate.
- **Syntax:** `SELECT * FROM table WHERE col > (SELECT AVG(col) FROM table)`
- **Dataset Application & Code Demonstration:** Finds transactions whose amount exceeds the overall system average transaction amount.


In [2]:
%%sql
SELECT 
    transaction_id,
    customer_id,
    transaction_amount
FROM transactions
WHERE transaction_amount > (SELECT AVG(transaction_amount) FROM transactions)
ORDER BY transaction_amount DESC
LIMIT 5;


,transaction_id,customer_id,transaction_amount
0,TX113073,C22899,1999.98
1,TX101707,C30635,1999.85
2,TX108075,C22224,1999.74
3,TX110393,C80575,1999.52
4,TX113287,C64524,1999.41


### 🔹 Multi-Value Subqueries: `IN (SELECT ...)`
- **What it does:** Tests whether a column value matches any element in a list produced by an inner subquery.
- **Syntax:** `SELECT * FROM table_1 WHERE key IN (SELECT key FROM table_2 WHERE condition)`
- **Dataset Application & Code Demonstration:** Selects all transactions executed by customers belonging to the `'PLATINUM'` tier.


In [3]:
%%sql
SELECT 
    transaction_id,
    customer_id,
    transaction_amount,
    card_type
FROM transactions
WHERE customer_id IN (
    SELECT customer_id 
    FROM customers 
    WHERE account_tier = 'PLATINUM'
)
LIMIT 5;


,transaction_id,customer_id,transaction_amount,card_type


### 🔹 Correlated Subqueries: Row-by-Row Outer Evaluation
- **What it does:** Executes an inner query that explicitly references attributes from the current outer row, re-evaluating for each candidate outer row.
- **Syntax:** `SELECT t1.* FROM t1 WHERE t1.val > (SELECT AVG(t2.val) FROM t2 WHERE t2.grp = t1.grp)`
- **Dataset Application & Code Demonstration:** Finds transactions that are greater than the average transaction amount for *their specific card type*.


In [4]:
%%sql
SELECT 
    t.transaction_id,
    t.card_type,
    t.transaction_amount
FROM transactions t
WHERE t.transaction_amount > (
    SELECT AVG(t_sub.transaction_amount)
    FROM transactions t_sub
    WHERE t_sub.card_type = t.card_type
)
LIMIT 5;


,transaction_id,card_type,transaction_amount
0,TX106376,Visa,1819.11
1,TX110701,Amex,1025.73
2,TX104105,Amex,1070.66
3,TX114584,Discover,1320.66
4,TX114810,Visa,1347.73


### 🔹 High-Performance Existence Validation: `EXISTS`
- **What it does:** Returns `TRUE` as soon as the inner subquery encounters the first matching tuple, short-circuiting further scan work.
- **Syntax:** `SELECT * FROM table_1 t1 WHERE EXISTS (SELECT 1 FROM table_2 t2 WHERE t2.key = t1.key)`
- **Dataset Application & Code Demonstration:** Selects customers who have at least one dispute recorded in the system.


In [5]:
%%sql
SELECT 
    c.customer_id,
    (c.first_name || ' ' || c.last_name) AS customer_name,
    c.account_tier
FROM customers c
WHERE EXISTS (
    SELECT 1 
    FROM disputes d
    INNER JOIN transactions t ON d.transaction_id = t.transaction_id
    WHERE t.customer_id = c.customer_id
)
LIMIT 5;


,customer_id,customer_name,account_tier
0,C93810,Richard Sharma,Platinum
1,C24592,Mark Scott,VIP
2,C13278,Karen Walker,Standard
3,C46048,Barbara Wilson,Silver
4,C42098,Daniel Thompson,Standard


## 💡 Real-World Practice & Scenarios
Practical scenarios and common data engineering questions explained with real examples.


### 🔍 Scenario: Q1: Correlated Subquery vs Window Function Performance
- **Objective:** Compare extracting regional benchmark differences via correlated subqueries vs window functions.
- **Approach:** Demonstrate how subquery unnesting achieves analytical parity with `AVG() OVER (PARTITION BY ...)`.


In [6]:
%%sql
SELECT 
    t.transaction_id,
    t.region,
    t.transaction_amount,
    ROUND(t.transaction_amount - (
        SELECT AVG(t2.transaction_amount) 
        FROM transactions t2 
        WHERE t2.region = t.region
    ), 2) AS diff_from_regional_mean
FROM transactions t
WHERE t.transaction_amount IS NOT NULL
LIMIT 5;


,transaction_id,region,transaction_amount,diff_from_regional_mean
0,TX109326,North,607.78,-394.49
1,TX106376,West,1819.11,810.04
2,TX103301,East,64.08,-930.62
3,TX110701,East,1025.73,31.03
4,TX103284,North,772.74,-229.53
